# Notebook 09 — Delay Discounting

Fits Mazur's (1987) hyperbolic discount function to participant choice data:

**V = A / (1 + k·D)**

Where V = subjective value, A = objective amount, D = delay, k = discount rate.

---
**Data requirements** (run `00_data_pipeline.ipynb` first):
- `data/processed/delay_discounting.parquet` — all DD events
- `references/level_specs/parameters.json` — condition parameters

**Sections**:
- A. Choice proportions P(LL) by delay condition
- B. Hyperbolic vs. exponential fit (per participant)
- C. Indifference point D₅₀ and k extraction
- D. Group discount curve + individual differences
- E. (Optional) Preference reversal test


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import curve_fit
from scipy import stats

DATA_DIR  = Path('../data/processed')
FIG_DIR   = Path('../reports/figures/delay_discounting')
SPECS_DIR = Path('../references/level_specs')
FIG_DIR.mkdir(parents=True, exist_ok=True)

with open(SPECS_DIR / 'parameters.json') as f:
    PARAMS = json.load(f)

DD = PARAMS['delay_discounting']
A_SS = DD['ss_amount_coins']   # 1
A_LL = DD['ll_amount_coins']   # 5
DELAYS = [c['ll_delay_s'] for c in DD['conditions']]
COND_LABELS = [c['condition'] for c in DD['conditions']]

df = pd.read_parquet(DATA_DIR / 'delay_discounting.parquet')
df['wall_time'] = pd.to_datetime(df['wall_time'])
print(f'Loaded {len(df):,} events across {df["participant_id"].nunique()} participants')
df.head()

## A. Choice Proportions P(LL) by Delay Condition

In [ ]:
def compute_p_ll(df):
    """For each participant × condition, compute P(LL choice) over trials."""
    rows = []
    for (pid, cond), grp in df.groupby(['participant_id', 'condition']):
        ss = (grp['event_code'] == 'SS_CHOICE').sum()
        ll = (grp['event_code'] == 'LL_CHOICE').sum()
        n  = ss + ll
        if n > 0:
            # Map condition string (e.g. 'D_20s') to delay in seconds
            delay = next((c['ll_delay_s'] for c in DD['conditions']
                          if c['condition'] == cond), None)
            rows.append({
                'participant_id': pid,
                'condition': cond,
                'delay_s': delay,
                'n_trials': n,
                'p_ll': ll / n,
                'n_ss': ss,
                'n_ll': ll,
            })
    return pd.DataFrame(rows)

choice = compute_p_ll(df)
group_summary = (choice
                 .groupby('delay_s')
                 .agg(mean_p_ll=('p_ll', 'mean'),
                      sem_p_ll =('p_ll', 'sem'),
                      n_participants=('participant_id', 'nunique'))
                 .reset_index())
group_summary

In [ ]:
# Plot group choice curve
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(group_summary['delay_s'], group_summary['mean_p_ll'],
            yerr=group_summary['sem_p_ll'],
            marker='o', markersize=10, linewidth=2, capsize=5,
            color='#e94560', ecolor='#334155')
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Indifference (P=0.5)')
ax.set_xlabel('LL delay (s)', fontsize=12)
ax.set_ylabel('P(LL choice)', fontsize=12)
ax.set_title('Delay Discounting — Group Choice Curve', fontweight='bold', fontsize=13)
ax.set_ylim(-0.05, 1.05)
ax.set_xscale('log')  # Hyperbolic on log-x often looks more linear
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'group_choice_curve.png', dpi=150)
plt.show()

## B. Hyperbolic vs. Exponential Fit (per participant)

Fit both models to each participant's P(LL, delay) data and compare.

In [ ]:
def hyperbolic_P(D, k):
    """P(LL) under hyperbolic discounting: P = V_LL / (V_LL + V_SS)
    V_LL = A_LL / (1 + k*D),  V_SS = A_SS (delay≈0)
    Returns softmax-style probability assuming choice ∝ V.
    """
    v_ll = A_LL / (1 + k * D)
    v_ss = A_SS
    return v_ll / (v_ll + v_ss)

def exponential_P(D, k):
    v_ll = A_LL * np.exp(-k * D)
    v_ss = A_SS
    return v_ll / (v_ll + v_ss)

def fit_individual(delays, p_ll, model='hyperbolic', k0=0.1):
    func = hyperbolic_P if model == 'hyperbolic' else exponential_P
    try:
        popt, _ = curve_fit(func, delays, p_ll, p0=[k0], bounds=(1e-4, 10))
        pred = func(np.array(delays), *popt)
        ss_res = np.sum((p_ll - pred) ** 2)
        ss_tot = np.sum((p_ll - p_ll.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
        # AIC: n·ln(SSR/n) + 2·k_params
        n = len(p_ll)
        aic = n * np.log(ss_res / n + 1e-10) + 2 * 1
        return popt[0], r2, aic
    except Exception:
        return np.nan, np.nan, np.nan

fits = []
for pid, grp in choice.groupby('participant_id'):
    g = grp.sort_values('delay_s')
    if len(g) < 3:
        continue
    k_hyp, r2_hyp, aic_hyp = fit_individual(g['delay_s'].values, g['p_ll'].values, 'hyperbolic')
    k_exp, r2_exp, aic_exp = fit_individual(g['delay_s'].values, g['p_ll'].values, 'exponential')
    fits.append({
        'participant_id': pid,
        'k_hyperbolic': k_hyp,   'r2_hyperbolic': r2_hyp,   'aic_hyperbolic': aic_hyp,
        'k_exponential': k_exp,  'r2_exponential': r2_exp,  'aic_exponential': aic_exp,
        'hyperbolic_better': aic_hyp < aic_exp,
    })

fit_df = pd.DataFrame(fits)
print(f"Hyperbolic fits better for {fit_df['hyperbolic_better'].sum()} / {len(fit_df)} participants")
fit_df.head()

In [ ]:
# Model comparison: R² distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist([fit_df['r2_hyperbolic'].dropna(), fit_df['r2_exponential'].dropna()],
             label=['Hyperbolic', 'Exponential'], bins=15, color=['#4ade80', '#f87171'], alpha=0.7)
axes[0].set_xlabel('R²')
axes[0].set_ylabel('Participants')
axes[0].set_title('Model Fit Quality (R²)')
axes[0].legend()

axes[1].scatter(fit_df['aic_exponential'], fit_df['aic_hyperbolic'], alpha=0.7, s=60)
lim = [min(fit_df['aic_hyperbolic'].min(), fit_df['aic_exponential'].min()),
       max(fit_df['aic_hyperbolic'].max(), fit_df['aic_exponential'].max())]
axes[1].plot(lim, lim, 'k--', alpha=0.5, label='Equal AIC')
axes[1].set_xlabel('AIC (exponential)')
axes[1].set_ylabel('AIC (hyperbolic)')
axes[1].set_title('AIC Comparison (lower = better)')
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'model_comparison.png', dpi=150)
plt.show()

## C. Indifference Point D₅₀ and k Extraction

The indifference delay D₅₀ satisfies V_LL = V_SS → A_LL/(1+k·D₅₀) = A_SS →
**k = (A_LL/A_SS − 1) / D₅₀ = 4 / D₅₀** (for A_LL=5, A_SS=1).

In [ ]:
# D50 from the fitted k:  k = 4/D50  →  D50 = 4/k
fit_df['D_50_s'] = (A_LL / A_SS - 1) / fit_df['k_hyperbolic']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fit_df['k_hyperbolic'].dropna(), bins=15, color='#4ade80', alpha=0.8, edgecolor='black')
axes[0].set_xlabel('k (hyperbolic discount rate)')
axes[0].set_ylabel('Participants')
axes[0].set_title('Individual Discount Rates (k)')
axes[0].axvline(fit_df['k_hyperbolic'].median(), color='red', linestyle='--',
                label=f"Median k = {fit_df['k_hyperbolic'].median():.3f}")
axes[0].legend()

axes[1].hist(fit_df['D_50_s'].dropna(), bins=15, color='#60a5fa', alpha=0.8, edgecolor='black')
axes[1].set_xlabel('Indifference point D₅₀ (s)')
axes[1].set_ylabel('Participants')
axes[1].set_title('Indifference Delay per Participant')
axes[1].axvline(fit_df['D_50_s'].median(), color='red', linestyle='--',
                label=f"Median D₅₀ = {fit_df['D_50_s'].median():.1f} s")
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'k_and_d50_distributions.png', dpi=150)
plt.show()

print('Discount rate k summary:')
print(fit_df['k_hyperbolic'].describe())
print('\nIndifference delay D₅₀ summary:')
print(fit_df['D_50_s'].describe())

## D. Group Discount Curve + Individual Differences

In [ ]:
# Plot all individual fits + group mean curve
fig, ax = plt.subplots(figsize=(9, 6))

D_plot = np.logspace(np.log10(1), np.log10(120), 200)

# Individual curves (thin, transparent)
for _, row in fit_df.iterrows():
    if not np.isnan(row['k_hyperbolic']):
        P = hyperbolic_P(D_plot, row['k_hyperbolic'])
        ax.plot(D_plot, P, color='#94a3b8', alpha=0.35, linewidth=1)

# Group-median curve (thick)
k_median = fit_df['k_hyperbolic'].median()
P_median = hyperbolic_P(D_plot, k_median)
ax.plot(D_plot, P_median, color='#e94560', linewidth=3,
        label=f'Group median curve (k={k_median:.3f})')

# Group empirical data points
ax.errorbar(group_summary['delay_s'], group_summary['mean_p_ll'],
            yerr=group_summary['sem_p_ll'],
            fmt='o', color='#0f3460', markersize=12, capsize=5,
            label='Group mean ± SEM (data)', zorder=10)

ax.axhline(0.5, color='gray', linestyle=':', alpha=0.6)
ax.set_xlabel('LL delay (s)', fontsize=12)
ax.set_ylabel('P(LL choice)', fontsize=12)
ax.set_title('Delay Discounting — Hyperbolic Model Fit (individual & group)',
             fontweight='bold', fontsize=13)
ax.set_xscale('log')
ax.set_ylim(-0.05, 1.05)
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(FIG_DIR / 'individual_and_group_curves.png', dpi=150)
plt.show()

In [ ]:
# Export fit results for downstream correlation with self-report or other measures
OUT_CSV = DATA_DIR / 'delay_discounting_fits.csv'
fit_df.to_csv(OUT_CSV, index=False)
print(f'Wrote {len(fit_df)} fits → {OUT_CSV}')

## E. (Optional) Preference Reversal Test

If near-pair (SS=1coin/1s, LL=5coin/31s) and distant-pair (SS=1coin/30s, LL=5coin/60s) 
conditions were collected, test whether the 30-s gap produces different preferences 
depending on absolute delay — hyperbolic's signature prediction.

In [ ]:
if 'near_pair' in df['condition'].unique() and 'distant_pair' in df['condition'].unique():
    reversal = compute_p_ll(df[df['condition'].isin(['near_pair', 'distant_pair'])])
    pivot = reversal.pivot(index='participant_id', columns='condition', values='p_ll')

    # Paired comparison
    if not pivot.empty:
        t, p = stats.ttest_rel(pivot['distant_pair'], pivot['near_pair'])
        print(f'Preference reversal paired t-test:')
        print(f't = {t:.3f}, p = {p:.4f}')
        print(f'Mean P(LL) near-pair:    {pivot["near_pair"].mean():.3f}')
        print(f'Mean P(LL) distant-pair: {pivot["distant_pair"].mean():.3f}')
        print('Hyperbolic prediction: distant > near (reversal toward LL when both distant).')

        fig, ax = plt.subplots(figsize=(7, 5))
        for pid in pivot.index:
            ax.plot(['near-pair', 'distant-pair'],
                    [pivot.loc[pid, 'near_pair'], pivot.loc[pid, 'distant_pair']],
                    '-o', alpha=0.5, color='#334155')
        ax.set_ylabel('P(LL choice)')
        ax.set_title('Preference Reversal (hyperbolic signature)', fontweight='bold')
        ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
        plt.tight_layout()
        plt.savefig(FIG_DIR / 'preference_reversal.png', dpi=150)
        plt.show()
else:
    print('Preference-reversal conditions not in data. Skipping.')

---
## Summary

- **Group discount curve**: P(LL) decreases with LL delay, matching hyperbolic prediction.
- **Individual k**: Extracted per participant; exported to `delay_discounting_fits.csv`.
- **Model comparison**: Hyperbolic typically fits better than exponential (AIC lower).
- **Indifference point D₅₀**: Median across participants provides group impulsivity index.

Individual k values can be correlated with clinical measures (BIS-11, substance use),
task-level variables (IOA, response times), or compared across participant groups.